# 06 — Held-out questions

Agent v2 was written after reading v1's failures on the main 100-question suite,
so its score there is not a clean measure. This notebook runs both versions on
40 questions that were written after v2 was finished (`eval/build_holdout.py`).
v2 has not been changed since.

The held-out set asks for the same skills in different words, mostly about
tickers that appear nowhere in the main suite, and adds question types the main
suite has no template for.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: vLLM + LMCache in their own virtualenv ---
# The server runs as a separate process, so it gets a separate environment.
# That keeps its packages from clashing with the ones Colab ships in the notebook.
VENV = "/content/venv-vllm"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python vllm lmcache
os.environ["SERVING_VENV"] = VENV

# Record the exact versions so the results can be reproduced.
!uv pip freeze --python {VENV}/bin/python | grep -iE "^(vllm|lmcache|torch|transformers)==" | tee results/versions_vllm.txt

In [ ]:
from serving.launch import start_server, stop_server

LARGE_MODEL = "Qwen/Qwen3-8B-AWQ"
SMALL_MODEL = "Qwen/Qwen3-0.6B"

## Large model

In [ ]:
server = start_server("vllm", LARGE_MODEL, log_name="holdout_large")
for version in ["v1", "v2"]:
    !python -m eval.run_eval --name holdout_{version}_8b --suite holdout --model {LARGE_MODEL} --prompt {version} --concurrency 16
stop_server(server)

## Small model

In [ ]:
server = start_server("vllm", SMALL_MODEL, log_name="holdout_small")
for version in ["v1", "v2"]:
    !python -m eval.run_eval --name holdout_{version}_0.6b --suite holdout --model {SMALL_MODEL} --prompt {version} --concurrency 16
stop_server(server)

## Results

In [ ]:
import json
import pandas as pd

summary = pd.read_csv("results/summary.csv")
display(summary[["name", "prompt", "questions", "correct", "accuracy_pct", "multi_step_correct", "errors", "wall_s"]])

# Correct answers per question group, one column per run
rows = {}
for name in summary["name"]:
    for line in open(f"results/{name}/records.jsonl"):
        record = json.loads(line)
        rows.setdefault(record["category"], {}).setdefault(name, 0)
        rows[record["category"]][name] += int(record["correct"])
pd.DataFrame(rows).T

In [ ]:
# --- Download the results ---
!zip -qr /content/results_06_holdout.zip results
from google.colab import files
files.download("/content/results_06_holdout.zip")